# Frozen AMASS-Core11 JEPA probes on GAVD

This experiment adapts the canonical 96 GAVD pose **sequences** (18 source videos) to the Core11 contract, freezes each checkpoint's EMA `target_encoder`, and compares nested ridge probes on trained, random, and raw-coordinate features. No encoder is retrained.

The task is five-class dataset-condition classification. It is a within-corpus descriptive readout, not unseen-video or clinical performance: all 12 normal sequences come from one source video, so five-class source-video-disjoint nested evaluation is impossible. The sequence-stratified folds below report their train/test video overlap explicitly.

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'src/gavd6_sjepa/research_directions/reflection_equivariance/gavd_core11_probe_evaluation.py').is_file():
            return candidate
    raise FileNotFoundError('Run inside the gavd6 repository')

PROJECT_DIR = find_project_root()
if str(PROJECT_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / 'src'))

from gavd6_sjepa.research_directions.reflection_equivariance.jepa_model_architecture import resolve_pose_dir
from gavd6_sjepa.research_directions.reflection_equivariance.gavd_core11_probe_evaluation import run_probe_experiment

POSE_DIR = resolve_pose_dir(PROJECT_DIR, os.getenv('GAIT_PARITY_POSE_DIR'))
CHECKPOINT_DIR = PROJECT_DIR / 'outputs/repaired-jepa-seed7-v2'
OUTPUT_DIR = PROJECT_DIR / 'work/artifacts/gavd_core11_frozen_probe'
DEVICE = os.getenv('GAIT_PARITY_DEVICE', 'cpu')
POSE_DIR, CHECKPOINT_DIR, DEVICE

## Frozen adapter and feature contract

MediaPipe indices 23–32 become bilateral hips, knees, ankles, heels, and forefeet; pelvis is the bilateral hip midpoint. MediaPipe foot-index is only a proxy for AMASS's big/small-toe midpoint. Full-frame x/z are in width units, so image y is aspect-corrected before defining the right-handed pseudo-world `[image right, camera depth, vertical up]`. The sequence-global AMASS frame algorithm is then reused with a leg-length-relative travel threshold. This is an AMASS-convention monocular pseudo-3D approximation, not metric AMASS reconstruction.

Coordinates are resampled at 30 Hz before 64-frame/32-stride windowing. Six short sequences are zero/invalid-padded rather than time-stretched. The transformer has no attention-padding mask: zero-invalid tokens can still affect contextual valid-token features before pooling. The requested all-96 result is therefore accompanied by a strict 90-sequence sensitivity that excludes every short padded clip and a validity-only nuisance probe. For each encoder, statistics are taken across paired-valid joint-time tokens from all windows; overlapping frames receive repeated window weight. `[even mean, even std, odd mean, odd std]` produces 256 features per sequence.

In [ ]:
result = run_probe_experiment(
    pose_dir=POSE_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    output_dir=OUTPUT_DIR,
    device=DEVICE,
    batch_size=32,
    seed=42,
)
print('sequences / source videos / windows:',
      len(result['adapted']),
      result['sequence_table'].video_id.nunique(),
      len(result['window_table']))

## Adapter coverage audit

Visibility, frame-selection method, and short-clip padding can carry condition-correlated information. The table separates raw observed coverage from post-interpolation/resampling usable coverage before any probe score is interpreted.

In [ ]:
display(result['condition_table'])
display(pd.crosstab(result['sequence_table'].condition, result['sequence_table'].frame_method))

## Identical nested ridge probes

Every representation uses the same five outer sequence-stratified folds and the same fold-local four-way inner alpha search. `StandardScaler` is fitted inside each training fold. Balanced accuracy is the selection and primary reporting metric; macro-F1 and accuracy are secondary. Random controls use seeds 7, 19, and 31 for each checkpoint-matched architecture.

In [ ]:
print('Requested all-96 zero/invalid-padded sensitivity')
display(result['summary'].sort_values('balanced_accuracy_mean', ascending=False))
print('Strict 90-sequence no-short-padding sensitivity')
display(result['strict_summary'].sort_values('balanced_accuracy_mean', ascending=False))
display(result['folds'].pivot(index='fold', columns='representation', values='balanced_accuracy'))
ema_rows = result['summary'][result['summary'].representation.str.startswith('ema_')]
random_rows = result['summary'][result['summary'].representation.str.startswith('random_')]
print('EMA balanced accuracy range:', tuple(ema_rows.balanced_accuracy_mean.round(3)))
print('Random-control balanced accuracy range:',
      (round(random_rows.balanced_accuracy_mean.min(), 3),
       round(random_rows.balanced_accuracy_mean.max(), 3)))
print('These descriptive folds do not show a trained-encoder advantage over random controls.')
print('Grouped five-class generalization: BLOCKED (normal has one source video).')
print('Artifacts:', result['output_dir'])